# Analyse de complexité — Problème de transport

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import glob
import os

## Chargement de tous les fichiers CSV

In [ ]:
# Récupère tous les CSV du dossier complexite
fichiers = glob.glob('complexite/*.csv')

# Vérification
if not fichiers:
    raise FileNotFoundError(
        "Aucun fichier CSV trouvé dans le dossier 'complexite'"
    )

# Lecture + concaténation
dfs = []

for fichier in fichiers:
    df_temp = pd.read_csv(fichier)

    # Ajoute le nom du fichier pour traçabilité
    df_temp['source'] = os.path.basename(fichier)

    dfs.append(df_temp)

# Fusion de tous les fichiers
df = pd.concat(dfs, ignore_index=True)

print(f"{len(fichiers)} fichiers chargés")
print(f"{len(df)} lignes au total")

# Aperçu
print(df.head())

## Préparation des données

In [ ]:
# Enveloppe supérieure du nuage de points
pire_des_cas = df.groupby('n').max(numeric_only=True)

print(pire_des_cas.head())

## Métriques étudiées

In [ ]:
metriques = {
    'thetaNO_ns': 'Temps Nord-Ouest (theta_NO)',
    'thetaBH_ns': 'Temps Balas-Hammer (theta_BH)',
    'tNO_ns': 'Temps Marche-pied depuis NO (t_NO)',
    'tBH_ns': 'Temps Marche-pied depuis BH (t_BH)',
    'totalNO_ns': 'Total (theta_NO + t_NO)',
    'totalBH_ns': 'Total (theta_BH + t_BH)'
}

## Nuage de points global + enveloppe supérieure

In [ ]:
plt.figure(figsize=(12, 9))

for col, label in metriques.items():

    # Nuage de points
    plt.scatter(
        df['n'],
        df[col],
        alpha=0.20,
        s=12,
        label=f'Données {label}'
    )

    # Pire des cas
    plt.plot(
        pire_des_cas.index,
        pire_des_cas[col],
        marker='o',
        linestyle='--'
    )

plt.xscale('log')
plt.yscale('log')

plt.xlabel('Taille du problème (n)')
plt.ylabel('Temps (ns)')

plt.title(
    'Analyse de la complexité temporelle\n'
    '(Tous fichiers fusionnés)'
)

plt.legend(fontsize=8)

plt.grid(
    True,
    which="both",
    ls="-",
    alpha=0.2
)

plt.tight_layout()
plt.show()

## Comparaison directe des algorithmes complets

In [ ]:
plt.figure(figsize=(12, 9))

plt.plot(
    pire_des_cas.index,
    pire_des_cas['totalNO_ns'],
    marker='s',
    linewidth=2,
    label='Pire des cas total NO'
)

plt.plot(
    pire_des_cas.index,
    pire_des_cas['totalBH_ns'],
    marker='s',
    linewidth=2,
    label='Pire des cas total BH'
)

plt.xscale('log')
plt.yscale('log')

plt.xlabel('Taille du problème (n)')
plt.ylabel('Temps total (ns)')

plt.title(
    'Comparaison Nord-Ouest vs Balas-Hammer\n'
    '(Tous fichiers fusionnés)'
)

plt.legend()

plt.grid(
    True,
    which="both",
    ls="-",
    alpha=0.2
)

plt.tight_layout()
plt.show()

## Analyse complémentaire : temps moyens

In [ ]:
moyennes = df.groupby('n').mean(numeric_only=True)

plt.figure(figsize=(12, 9))

plt.plot(
    moyennes.index,
    moyennes['totalNO_ns'],
    marker='o',
    label='Temps moyen total NO'
)

plt.plot(
    moyennes.index,
    moyennes['totalBH_ns'],
    marker='o',
    label='Temps moyen total BH'
)

plt.xscale('log')
plt.yscale('log')

plt.xlabel('Taille du problème (n)')
plt.ylabel('Temps moyen (ns)')

plt.title('Temps moyens des algorithmes')

plt.legend()

plt.grid(True, which="both", alpha=0.2)

plt.tight_layout()
plt.show()

## Nombre de cas ayant atteint maxIter

In [ ]:
if 'optimalNO' in df.columns and 'optimalBH' in df.columns:

    no_fail = (~df['optimalNO']).sum()
    bh_fail = (~df['optimalBH']).sum()

    print(f"Cas maxIter atteints (NO) : {no_fail}")
    print(f"Cas maxIter atteints (BH) : {bh_fail}")

## Calcul des pentes log log

In [ ]:
from scipy import stats

# Classes de complexité selon le tableau du sujet
def identifier_classe(pente):
    if pente < 0.3:
        return "O(log n)"
    elif pente < 1.3:
        return "O(n)"
    elif pente < 1.8:
        return "O(n log n)"
    elif pente < 2.5:
        return "O(n²)"
    elif pente < 3.5:
        return "O(n³)"
    else:
        return f"O(n^{pente:.1f})"

ns = pire_des_cas.index.astype(float).values
log_n = np.log(ns)

print("=" * 65)
print(f"{'Métrique':<30} {'Pente':>7}  {'R²':>6}  {'Classe'}")
print("=" * 65)

pentes = {}

for col, label in metriques.items():
    valeurs = pire_des_cas[col].values.astype(float)

    # Filtrage des valeurs nulles ou négatives (impossibles en log)
    masque = valeurs > 0
    if masque.sum() < 2:
        print(f"{label:<30}  données insuffisantes")
        continue

    slope, intercept, r, p, se = stats.linregress(log_n[masque], np.log(valeurs[masque]))
    classe = identifier_classe(slope)
    pentes[col] = slope

    print(f"{label:<30} {slope:>7.2f}  {r**2:>6.3f}  {classe}")

print("=" * 65)
print("R² proche de 1 = bon ajustement de la loi puissance.")

## Visualisation des droites d'ajustement log-log

In [ ]:
## Visualisation des droites d'ajustement log-log

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for ax, (col, label) in zip(axes, metriques.items()):
    valeurs = pire_des_cas[col].values.astype(float)
    masque = valeurs > 0

    # Nuage de points (tous les runs)
    for n_val in ns[masque]:
        points = df[df['n'] == n_val][col].values
        ax.scatter([n_val] * len(points), points, s=6, alpha=0.2, color='steelblue')

    # Courbe du pire cas
    ax.plot(ns[masque], valeurs[masque], 'D--', color='darkorange',
            markersize=6, linewidth=1.8, label='Pire cas')

    # Droite d'ajustement log-log
    if col in pentes:
        slope = pentes[col]
        log_n_fit = np.log(ns[masque])
        intercept_fit = np.mean(np.log(valeurs[masque]) - slope * log_n_fit)
        y_fit = np.exp(intercept_fit + slope * log_n_fit)
        ax.plot(ns[masque], y_fit, 'r-', linewidth=1.5,
                label=f'Ajustement : pente={slope:.2f}\n→ {identifier_classe(slope)}')

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title(label, fontsize=9, fontweight='bold')
    ax.set_xlabel('n')
    ax.set_ylabel('Temps (ns)')
    ax.legend(fontsize=7)
    ax.grid(True, which='both', alpha=0.2)

plt.suptitle('Ajustement log-log par métrique — Identification des classes de complexité',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Tableau récapitulatif final

In [ ]:
print("\nRÉCAPITULATIF — Classes de complexité dans le pire des cas")
print("=" * 50)
for col, label in metriques.items():
    if col in pentes:
        p = pentes[col]
        print(f"  {label}")
        print(f"    → pente = {p:.2f}  |  classe ≈ {identifier_classe(p)}")
        print()
